In [ ]:
from ruamel.yaml import YAML
import numpy as np

import torch
import torch.backends.cudnn as cudnn

from transformers import BertTokenizerFast
from MMDF_ import utils
from MMDF_.dataset import create_dataset, create_loader
from MMDF_.scheduler import create_scheduler
from MMDF_.optim import create_optimizer

from tqdm import tqdm

from MMDF_.models.HAMMER import HAMMER
from MMDF_.utils import text_input_adjust, AttrDict, evaluate, load_checkpoint
from collections import OrderedDict
import os
from accelerate import Accelerator

In [ ]:
args = {
    'config': r'MMDF_\configs\train.yaml',
    'checkpoint': '',
    'resume': False,
    'output_dir': 'results',
    'text_encoder': 'bert-base-uncased',
    'device': 'cuda',
    'world_size': 1,
    'launcher': 'pytorch',
    'model_save_epoch': 1,
    'token_momentum': True
}
args = AttrDict(args)
yaml = YAML(typ='safe') 

with open(args.config, 'r') as f:
    config = yaml.load(f)

In [ ]:
def train(args, model, data_loader, optimizer, tokenizer, epoch, EPOCH, warmup_steps, device, scheduler, config, accelerator):
    # train
    model.train()  
    
    print_freq = 100   
    step_size = 100
    warmup_iterations = warmup_steps * step_size  

    global_step = epoch * len(data_loader)
    avg_loss = []

    # text = caption; fake_word_pos = fake_text_pos_list
    pbar = tqdm(data_loader)
    for i, (image, label, text, fake_image_box, fake_word_pos, W, H) in enumerate(pbar):
        # customize pbar
        pbar.set_description(f"Epoch {epoch}/{EPOCH}")

        if config['schedular']['sched'] == 'cosine_in_step':
            scheduler.adjust_learning_rate(optimizer, i / len(data_loader) + epoch, args, config)        

        # zero gradient
        optimizer.zero_grad()
  
        # move to device
        image = image.to(device, non_blocking=True)
        fake_image_box = fake_image_box.to(device)
        
        text_input = tokenizer(text, max_length=128, truncation=True, add_special_tokens=True, return_attention_mask=True, return_token_type_ids=False) 
        
        # forward
        text_input, fake_token_pos = text_input_adjust(text_input, fake_word_pos, device) # text input is list token
 
        if epoch > 0:
            alpha = config['alpha']
        else:
            alpha = config['alpha'] * min(1, i / len(data_loader)) 
        
        loss_MAC, loss_BIC, loss_bbox, loss_giou, loss_TMG, loss_MLC = model(image, label, text_input, fake_image_box, fake_token_pos, alpha=alpha)  
            
        loss = config['loss_MAC_wgt'] * loss_MAC \
             + config['loss_BIC_wgt'] * loss_BIC \
             + config['loss_bbox_wgt'] * loss_bbox \
             + config['loss_giou_wgt'] * loss_giou \
             + config['loss_TMG_wgt'] * loss_TMG \
             + config['loss_MLC_wgt'] * loss_MLC \
                 
        avg_loss.append(loss.item())
        
        # backward
        # loss.backward()
        accelerator.backward(loss)
        optimizer.step()    
        
        # customize pbar
        pbar.set_postfix(OrderedDict([
            ('MAC', f'{loss_MAC.item():.3f}'),
            ('BIC', f'{loss_BIC.item():.3f}'),
            ('bbox', f'{loss_bbox.item():.3f}'),
            ('giou', f'{loss_giou.item():.3f}'),
            ('TMG', f'{loss_TMG.item():.3f}'),
            ('MLC', f'{loss_MLC.item():.3f}'),
            ('loss', f'{loss.item():.3f}'),
            ('AVG', f'{np.mean(avg_loss):.3f}'),
            ('lr', f'{optimizer.param_groups[0]["lr"]}')
        ]))
        
        if epoch == 0 and i % step_size==0 and i <= warmup_iterations and config['schedular']['sched'] != 'cosine_in_step': 
            scheduler.step(i // step_size)   

        global_step += 1
        break

In [ ]:
# device = torch.device(args.device)
accelerator = Accelerator()
device = accelerator.device
cudnn.benchmark = True

start_epoch = 0
max_epoch = config['schedular']['epochs']  # 50
warmup_steps = config['schedular']['warmup_epochs']  # 100

#### Dataset #### 
train_dataset, val_dataset = create_dataset(config)


train_loader, val_loader = create_loader([train_dataset, val_dataset],
                            batch_size=[config['batch_size_train']] + [config['batch_size_val']], 
                            num_workers=[4, 4],
                            is_trains=[True, False]
)

# LOAD BERT TOKENIZER
tokenizer = BertTokenizerFast.from_pretrained(args.text_encoder)

#### Model #### 
model = HAMMER(args=args, config=config, text_encoder=args.text_encoder, tokenizer=tokenizer, init_deit=True)
model = model.to(device)

In [ ]:
# optimizer
arg_opt = utils.AttrDict(config['optimizer'])
optimizer = create_optimizer(arg_opt, model)
# scheduler
arg_sche = utils.AttrDict(config['schedular'])
lr_scheduler, _ = create_scheduler(arg_sche, optimizer)
if config['schedular']['sched'] == 'cosine_in_step':
    args.lr = config['optimizer']['lr']
# load checkpoint
load_checkpoint(args, model, optimizer, lr_scheduler)

In [ ]:
model, optimizer, train_loader, val_loader, lr_scheduler = accelerator.prepare(
        model, optimizer, train_loader, val_loader, lr_scheduler
    )

In [ ]:
for epoch in range(start_epoch, 3):
    # train
    print("train\n")
    train(args, model, train_loader, optimizer, tokenizer, epoch+1, max_epoch, warmup_steps, device, lr_scheduler, config, accelerator)
    # val
    print("val\n")
    val_stats = evaluate(args, model, val_loader, tokenizer, optimizer, lr_scheduler, epoch+1, warmup_steps, device, config)
    os.makedirs('results', exist_ok=True)
    with open(os.path.join('results', f"result_{epoch+1}.txt"), "w") as f:
        f.write(str(val_stats))